# Export verified labels

Reads each verified transcript from `data/question_labels/verified/` (`*_qid_verified.xlsx`),
keeps only the `role`, `text`, and `verified_qid` columns, renames `verified_qid` -> `qid`,
and writes `{SUBJECT_ID}_qid.xlsx` to `data/transcripts_german`.

In [20]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(Path.cwd()).resolve().parent.parent
VERIFIED_DIR = PROJECT_ROOT / "data" / "question_labels" / "verified"
OUTPUT_DIR   = PROJECT_ROOT / "data" / "transcripts_german"

KEEP_COLUMNS = ["role", "text", "verified_qid"]
CUTOFF_QID = "Q5"  # drop the first Q5 row and everything after it (to retain only "identity questions"); In real experiment this is Q16.
IDS_TO_INCLUDE = ["Q1", "Q2", "Q3", "Q4"]   # qids that must be present
#print("Project root:", PROJECT_ROOT)

In [21]:
# ── Sanity check are all QIDS present, it CUTOFF_QID present & comes after TO_INCLUDE──────────────────

for file_path in sorted(VERIFIED_DIR.glob("*_qid_verified.xlsx")):
    subject_id = file_path.stem.replace("_qid_verified", "")

    df = pd.read_excel(file_path, engine="openpyxl")
    df.columns = [c.lower() for c in df.columns]

    if "verified_qid" not in df.columns:
        print(f"{subject_id}: missing 'verified_qid' column — skipped")
        continue

    # Same normalize + forward-fill as the export cell.
    qid = (
        df["verified_qid"]
        .replace(r"^\s*$", pd.NA, regex=True)
        .ffill()
        .astype("string")
        .str.strip()
    )

    def first_pos(value):
        mask = (qid == value).fillna(False).to_numpy()
        return int(mask.argmax()) if mask.any() else None

    problems = []

    # 1) any wanted IDs missing entirely
    missing = [q for q in IDS_TO_INCLUDE if first_pos(q) is None]
    if missing:
        problems.append(f"missing: {missing}")

    # 2) is the cutoff present?
    cutoff_pos = first_pos(CUTOFF_QID)
    if cutoff_pos is None:
        problems.append(f"cutoff '{CUTOFF_QID}' not found")
    else:
        # 3) any wanted ID whose first appearance is AFTER the cutoff
        #    (it would be dropped by the Q5 cut)
        after = [
            q for q in IDS_TO_INCLUDE
            if (p := first_pos(q)) is not None and p > cutoff_pos
        ]
        if after:
            problems.append(f"appear after '{CUTOFF_QID}' (would be cut): {after}")

    print(f"{subject_id}: {'OK' if not problems else '; '.join(problems)}")

T0001: OK
T0002: OK
T0003: OK
T0004: OK
T0005: OK


In [22]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

verified_files = sorted(VERIFIED_DIR.glob("*_qid_verified.xlsx"))
print(f"Found {len(verified_files)} verified file(s) in {VERIFIED_DIR}\n")

for file_path in verified_files:
    subject_id = file_path.stem.replace("_qid_verified", "")

    df = pd.read_excel(file_path, engine="openpyxl")
    df.columns = [c.lower() for c in df.columns]  # normalize column names

    missing = [c for c in KEEP_COLUMNS if c not in df.columns]
    if missing:
        print(f"Skipping {file_path.name}: missing columns {missing}")
        continue

    out = df[KEEP_COLUMNS].rename(columns={"verified_qid": "qid"})

    # Forward-fill qid: each labeled qid carries down to every following
    # row until the next qid appears. Blank strings are treated as missing.
    out["qid"] = (
        out["qid"]
        .replace(r"^\s*$", pd.NA, regex=True)
        .ffill()
    )

    # Drop the leading rows that never received a qid (those before the
    # first labeled question).
    out = out[out["qid"].notna()]

    # Q5 is the cutoff: drop the first CUTOFF_QID row and everything after it.
    is_q5 = out["qid"].astype(str).str.strip() == CUTOFF_QID
    if is_q5.any():
        cutoff = is_q5.to_numpy().argmax()   # position of the first Q5 row
        out = out.iloc[:cutoff]

    out = out.reset_index(drop=True)

    out_path = OUTPUT_DIR / f"{subject_id}_qid.xlsx"
    out.to_excel(out_path, index=False)
    print(f"Wrote {out_path.name}  ({len(out)} rows)")

print("\nDone.")

Found 5 verified file(s) in /Users/katra/Library/Mobile Documents/com~apple~CloudDocs/Desktop/WORKSPACE/splitting/splitting_paper/toshare/data/question_labels/verified

Wrote T0001_qid.xlsx  (24 rows)
Wrote T0002_qid.xlsx  (16 rows)
Wrote T0003_qid.xlsx  (34 rows)
Wrote T0004_qid.xlsx  (20 rows)
Wrote T0005_qid.xlsx  (24 rows)

Done.
